# RQ4 Contextual Moderation Main Analysis

This notebook estimates RQ4 cross-level moderation models using GDL-level observations and national context moderators. It intentionally does **not** use country fixed effects, because national context variables are country-level. Standard errors are clustered by country. Formal outputs are written to `Results/RQ4/Tables` and `Results/RQ4/Figures`; process files are written to `Process/RQ4_main`.

In [ ]:

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import statsmodels.api as sm

warnings.filterwarnings('ignore')

project = Path('/Users/nico/Research/Energy sufficiency')
data_path = project / 'Data/global_SE_EC_sufficiency_controls_nationalindex_2015.csv'
results_dir = project / 'Results/RQ4'
tables_dir = results_dir / 'Tables'
figures_dir = results_dir / 'Figures'
process_dir = project / 'Process/RQ4_main'
for d in [tables_dir, figures_dir, process_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Keep pretest separate; clean only formal RQ4 top-level outputs.
for folder in [tables_dir, figures_dir]:
    for f in folder.glob('*'):
        if f.is_file():
            f.unlink()

palette_path = project / 'Process/rq1_region_palette.csv'
region_palette = pd.read_csv(palette_path)
region_color = dict(zip(region_palette['analytical_region'], region_palette['color_hex']))
region_short = dict(zip(region_palette['analytical_region'], region_palette['short_label']))

print('Input:', data_path)
print('Formal outputs:', results_dir)


## 1. Variable Construction

RQ4 follows the RQ3 M4 social predictor set: SES, rurality, education, and female share. SES follows the project convention: use `income` when valid, otherwise use `wealth`. The four national context moderators are GII, income inequality, education inequality, and national rural population rate.

In [ ]:

raw = pd.read_csv(data_path)
d = raw.copy()

# RQ2/RQ3 convention.
d['income_clean'] = d['income'].where(d['income'].notna() & (d['income'] >= 0))
d['wealth_clean'] = d['wealth'].where(d['wealth'].notna() & (d['wealth'] >= 0))
d['SES_raw'] = d['income_clean'].combine_first(d['wealth_clean'])
d['female_share'] = d['gender']

# GDL-level controls, selected after pretest.
d['GDP_pc'] = d['GDP'] / d['population'].where(d['population'] > 0)
d['log_GDP_pc'] = np.log(d['GDP_pc'].where(d['GDP_pc'] > 0))
d['log_population'] = np.log(d['population'].where(d['population'] > 0))

# Main outcome.
d['log_sufficiency_ratio'] = np.log(d['sufficiency_ratio'].where(d['sufficiency_ratio'] > 0))

outcome = 'log_sufficiency_ratio'
country_col = 'iso_code'
region_col = 'analytical_region'
social_vars = ['SES_raw', 'rural', 'education', 'female_share']
context_vars = ['national_GII', 'national_inequality_income', 'national_inequality_education', 'national_urbanisation_rate']
controls = ['log_GDP_pc', 'log_population', 'total_HDD', 'total_CDD']
interaction_defs = [
    {'term': 'int_female_GII', 'x': 'female_share', 'z': 'national_GII', 'label': 'Female share x GII'},
    {'term': 'int_SES_income_ineq', 'x': 'SES_raw', 'z': 'national_inequality_income', 'label': 'SES x income inequality'},
    {'term': 'int_education_edu_ineq', 'x': 'education', 'z': 'national_inequality_education', 'label': 'Education x education inequality'},
    {'term': 'int_rural_urbanisation', 'x': 'rural', 'z': 'national_urbanisation_rate', 'label': 'Rurality x urbanisation rate'},
]
labels = {
    'SES_raw': 'SES', 'rural': 'Rurality', 'education': 'Education', 'female_share': 'Female share',
    'national_GII': 'GII', 'national_inequality_income': 'Income inequality',
    'national_inequality_education': 'Education inequality', 'national_rural_population_rate': 'Rural population rate',
    'log_GDP_pc': 'GDP per capita', 'log_population': 'Population', 'total_HDD': 'HDD', 'total_CDD': 'CDD',
    'int_female_GII': 'Female share x GII', 'int_SES_income_ineq': 'SES x income inequality',
    'int_education_edu_ineq': 'Education x education inequality', 'int_rural_urbanisation': 'Rurality x urbanisation rate',
}

required = [outcome, country_col, region_col] + social_vars + context_vars + controls
sample = d[d[required].notna().all(axis=1)].copy()

# Standardize using the global complete sample; region models use the same scaling for comparability.
standardized_vars = social_vars + context_vars + controls
standardization = []
for v in standardized_vars:
    mean = sample[v].mean()
    sd = sample[v].std(ddof=0)
    sample[f'z_{v}'] = (sample[v] - mean) / sd
    d[f'z_{v}'] = (d[v] - mean) / sd
    standardization.append({'variable': v, 'mean': mean, 'sd': sd})
std_table = pd.DataFrame(standardization)
std_table.to_csv(process_dir / 'rq4_main_standardization_parameters.csv', index=False)

for item in interaction_defs:
    term = item['term']
    x = item['x']
    z = item['z']
    sample[term] = sample[f'z_{x}'] * sample[f'z_{z}']

sample.to_csv(process_dir / 'rq4_main_common_sample.csv', index=False)

sample_summary = pd.DataFrame([
    {'sample': 'RQ4 formal common sample', 'gdl_rows': len(sample), 'countries': sample[country_col].nunique(), 'regions': sample[region_col].nunique()},
])
region_summary = sample.groupby(region_col).agg(
    gdl_rows=('GDLcode', 'count'),
    countries=(country_col, 'nunique'),
    mean_gdl_per_country=('GDLcode', lambda s: len(s) / sample.loc[s.index, country_col].nunique()),
).reset_index()
region_summary['cluster_warning'] = np.select(
    [region_summary['countries'] < 10, region_summary['countries'] < 20],
    ['high risk: <10 countries', 'caution: 10-19 countries'],
    default='ok'
)

sample_summary.to_csv(tables_dir / 'Table_RQ4_0_sample_summary.csv', index=False)
region_summary.to_csv(tables_dir / 'Table_RQ4_0_region_sample_summary.csv', index=False)
print(sample_summary.to_string(index=False))
print(region_summary.to_string(index=False))


## 2. Model Functions

The global model includes region fixed effects and country-clustered standard errors. Region-specific models are estimated within each region and therefore include no region fixed effects. All continuous predictors and moderators are standardized before interaction construction.

In [ ]:

def bh_fdr(p_values):
    p = pd.Series(p_values, dtype=float)
    q = pd.Series(np.nan, index=p.index, dtype=float)
    valid = p.dropna().sort_values()
    m = len(valid)
    if m == 0:
        return q
    ranked = valid.copy()
    prev = 1.0
    adjusted = {}
    for rank, (idx, val) in enumerate(valid.iloc[::-1].items(), start=1):
        true_rank = m - rank + 1
        adj = min(prev, val * m / true_rank)
        adjusted[idx] = min(adj, 1.0)
        prev = adj
    for idx, val in adjusted.items():
        q.loc[idx] = val
    return q

def sig_stars(p):
    if pd.isna(p):
        return ''
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    if p < 0.1:
        return '†'
    return ''

def build_design(df, include_region_fe=False):
    cols = [f'z_{v}' for v in social_vars + context_vars + controls] + [item['term'] for item in interaction_defs]
    X = df[cols].copy()
    if include_region_fe:
        X = pd.concat([X, pd.get_dummies(df[region_col], prefix='region', drop_first=True, dtype=float)], axis=1)
    X = sm.add_constant(X, has_constant='add')
    return X

def fit_model(df, include_region_fe, model_name):
    X = build_design(df, include_region_fe=include_region_fe)
    y = df[outcome]
    cluster_note = 'country-clustered'
    try:
        model = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': df[country_col]})
    except Exception as exc:
        model = sm.OLS(y, X).fit(cov_type='HC3')
        cluster_note = f'HC3 fallback: {type(exc).__name__}'
    return model, cluster_note

def coefficient_table(model, df, model_name, include_region_terms=False, cluster_note='country-clustered'):
    rows = []
    for var in model.params.index:
        if var == 'const':
            continue
        if (not include_region_terms) and var.startswith('region_'):
            continue
        base = var[2:] if var.startswith('z_') else var
        rows.append({
            'model': model_name,
            'variable': base,
            'label': labels.get(base, labels.get(var, var)),
            'coef': model.params[var],
            'se': model.bse[var],
            'p_value': model.pvalues[var],
            'stars': sig_stars(model.pvalues[var]),
            'n_obs': int(model.nobs),
            'n_countries': int(df[country_col].nunique()),
            'r_squared': float(model.rsquared),
            'adj_r_squared': float(model.rsquared_adj),
            'se_type': cluster_note,
        })
    return pd.DataFrame(rows)


## 3. Global Cross-Level Moderation Model

In [ ]:

global_model, global_note = fit_model(sample, include_region_fe=True, model_name='Global_RQ4_region_FE')
global_coef = coefficient_table(global_model, sample, 'Global_RQ4_region_FE', include_region_terms=False, cluster_note=global_note)
interaction_terms = [item['term'] for item in interaction_defs]
mask_int = global_coef['variable'].isin(interaction_terms)
global_coef.loc[mask_int, 'q_value_global_4_interactions'] = bh_fdr(global_coef.loc[mask_int, 'p_value']).values
global_coef['q_stars_global_4_interactions'] = global_coef['q_value_global_4_interactions'].apply(sig_stars)
# Effect interpretation for non-interaction 1 SD terms.
global_coef['effect_pct_per_1SD'] = np.where(
    global_coef['variable'].isin([v for v in social_vars + context_vars + controls]),
    (np.exp(global_coef['coef']) - 1) * 100,
    np.nan
)
global_coef.to_csv(tables_dir / 'Table_RQ4_1_global_contextual_moderation_coefficients.csv', index=False)

global_interactions = global_coef[global_coef['variable'].isin(interaction_terms)].copy()
global_interactions['interaction_family'] = 'global four cross-level interactions'
global_interactions.to_csv(tables_dir / 'Table_RQ4_2_global_interactions_FDR.csv', index=False)

print(global_interactions[['label','coef','se','p_value','q_value_global_4_interactions','stars','q_stars_global_4_interactions','n_obs','n_countries']].to_string(index=False))


## 4. Region-Specific Moderation Models

In [ ]:

region_coef_tables = []
region_fit_rows = []
for region, sub in sample.groupby(region_col, sort=False):
    model_name = f'Region_RQ4_{region}'
    model, note = fit_model(sub, include_region_fe=False, model_name=model_name)
    ct = coefficient_table(model, sub, model_name, include_region_terms=False, cluster_note=note)
    ct['analytical_region'] = region
    ct['region_short_label'] = region_short.get(region, region)
    ct['cluster_warning'] = 'high risk: <10 countries' if sub[country_col].nunique() < 10 else ('caution: 10-19 countries' if sub[country_col].nunique() < 20 else 'ok')
    region_coef_tables.append(ct)
    region_fit_rows.append({
        'analytical_region': region,
        'gdl_rows': int(len(sub)),
        'countries': int(sub[country_col].nunique()),
        'r_squared': float(model.rsquared),
        'adj_r_squared': float(model.rsquared_adj),
        'se_type': note,
        'cluster_warning': ct['cluster_warning'].iloc[0],
    })

region_coef = pd.concat(region_coef_tables, ignore_index=True)
region_fit = pd.DataFrame(region_fit_rows)
region_interactions = region_coef[region_coef['variable'].isin(interaction_terms)].copy()
# FDR within each region across four interactions.
region_interactions['q_value_within_region_4'] = np.nan
for region, idx in region_interactions.groupby('analytical_region').groups.items():
    region_interactions.loc[idx, 'q_value_within_region_4'] = bh_fdr(region_interactions.loc[idx, 'p_value']).values
# FDR across all 28 region-interaction tests.
region_interactions['q_value_all_region_28'] = bh_fdr(region_interactions['p_value']).values
region_interactions['q_stars_within_region_4'] = region_interactions['q_value_within_region_4'].apply(sig_stars)
region_interactions['q_stars_all_region_28'] = region_interactions['q_value_all_region_28'].apply(sig_stars)

region_coef.to_csv(tables_dir / 'Table_RQ4_3_region_model_coefficients.csv', index=False)
region_interactions.to_csv(tables_dir / 'Table_RQ4_4_region_interactions_FDR.csv', index=False)
region_fit.to_csv(tables_dir / 'Table_RQ4_5_region_model_fit_summary.csv', index=False)

print(region_fit.to_string(index=False))
print(region_interactions[['analytical_region','label','coef','se','p_value','q_value_within_region_4','q_value_all_region_28','cluster_warning']].sort_values('p_value').head(15).to_string(index=False))


## 5. Visualisation

In [ ]:

plt.rcParams.update({
    'font.size': 13,
    'axes.titlesize': 17,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
})

panel_terms = [item['term'] for item in interaction_defs]
plot_rows = []
for item in interaction_defs:
    term = item['term']
    grow = global_interactions[global_interactions['variable'].eq(term)].iloc[0]
    plot_rows.append({
        'panel': item['label'], 'unit': 'Global', 'unit_label': 'Global',
        'coef': grow['coef'], 'se': grow['se'], 'p_value': grow['p_value'],
        'q_value': grow['q_value_global_4_interactions'], 'stars': sig_stars(grow['q_value_global_4_interactions']),
        'color': '#333333', 'n_countries': grow['n_countries'], 'warning': '', 'is_global': True,
    })
    for region in list(region_palette['analytical_region']):
        rrow = region_interactions[(region_interactions['analytical_region'].eq(region)) & (region_interactions['variable'].eq(term))].iloc[0]
        plot_rows.append({
            'panel': item['label'], 'unit': region, 'unit_label': region_short.get(region, region).replace('\n', ' '),
            'coef': rrow['coef'], 'se': rrow['se'], 'p_value': rrow['p_value'],
            'q_value': rrow['q_value_all_region_28'], 'stars': sig_stars(rrow['q_value_all_region_28']),
            'color': region_color.get(region, '#999999'), 'n_countries': rrow['n_countries'],
            'warning': rrow['cluster_warning'], 'is_global': False,
        })
plot_df = pd.DataFrame(plot_rows)
plot_df.to_csv(tables_dir / 'Table_RQ4_6_plotting_global_region_interaction_coefficients.csv', index=False)

fig, axes = plt.subplots(2, 2, figsize=(17.0, 12.6), sharex=False)
axes = axes.ravel()
unit_order = ['Global'] + list(region_palette['analytical_region'])

for ax, item in zip(axes, interaction_defs):
    panel = item['label']
    sub = plot_df[plot_df['panel'].eq(panel)].copy()
    sub['unit_order'] = sub['unit'].apply(lambda u: unit_order.index(u))
    sub = sub.sort_values('unit_order')
    sub['lo'] = sub['coef'] - 1.96 * sub['se']
    sub['hi'] = sub['coef'] + 1.96 * sub['se']
    y_positions = np.arange(len(sub))[::-1]

    stable = sub[~((sub['warning'].astype(str).str.contains('high risk')) & ((sub['hi'] - sub['lo']) > 8))]
    if stable.empty:
        stable = sub
    xmin = stable['lo'].min()
    xmax = stable['hi'].max()
    pad = max((xmax - xmin) * 0.18, 0.30)
    xmin_plot = xmin - pad
    xmax_plot = xmax + pad * 1.25
    ax.set_xlim(xmin_plot, xmax_plot)
    ax.axvline(0, color='0.35', linestyle='--', linewidth=1.1)

    for y, (_, row) in zip(y_positions, sub.iterrows()):
        lo, hi, coef = row['lo'], row['hi'], row['coef']
        lo_clip = max(lo, xmin_plot)
        hi_clip = min(hi, xmax_plot - pad * 0.18)
        coef_clip = min(max(coef, xmin_plot), xmax_plot - pad * 0.18)
        clipped = (lo < xmin_plot) or (hi > xmax_plot - pad * 0.18) or (coef != coef_clip)
        alpha = 1.0 if row['is_global'] or row['stars'] else 0.72
        face = row['color']
        edge = 'white'
        lw = 1.4
        if 'high risk' in str(row['warning']):
            face = 'white'
            edge = row['color']
            lw = 2.0
        ax.hlines(y, lo_clip, hi_clip, color=row['color'] if row['is_global'] or row['stars'] else '#b6c5d0', linewidth=2.5, alpha=alpha)
        marker = '>' if coef > xmax_plot else ('<' if coef < xmin_plot else 'o')
        ax.scatter(coef_clip, y, s=112 if row['is_global'] else 98, marker=marker, facecolor=face, edgecolor=edge, linewidth=lw, zorder=3, alpha=alpha)
        if clipped:
            ax.text(hi_clip if hi > xmax_plot else lo_clip, y + 0.23, 'CI clipped', fontsize=8.8, color='0.35', ha='right' if hi > xmax_plot else 'left', va='center')
        if row['stars']:
            # Put significance next to its own estimate, nudged away from the CI direction and kept inside panel bounds.
            if coef_clip >= 0:
                x_anno = min(max(hi_clip + pad * 0.07, coef_clip + pad * 0.10), xmax_plot - pad * 0.20)
                ha = 'left'
            else:
                x_anno = max(min(lo_clip - pad * 0.07, coef_clip - pad * 0.10), xmin_plot + pad * 0.20)
                ha = 'right'
            ax.text(x_anno, y + 0.18, row['stars'], color='#c9252d', fontsize=16.5, fontweight='bold', va='center', ha=ha)
            ax.text(x_anno, y - 0.12, f"{row['coef']:+.2f}, q={row['q_value']:.3f}", color='0.12', fontsize=11.2, fontweight='bold', va='center', ha=ha)

    labels_y = []
    for _, row in sub.iterrows():
        if row['unit'] == 'Global':
            lab = 'Global'
        elif 'high risk' in str(row['warning']):
            lab = f"{row['unit_label']}\n(n={int(row['n_countries'])}, high risk)"
        elif 'caution' in str(row['warning']):
            lab = f"{row['unit_label']}\n(n={int(row['n_countries'])}, caution)"
        else:
            lab = f"{row['unit_label']}\n(n={int(row['n_countries'])})"
        labels_y.append(lab)
    ax.set_yticks(y_positions, labels_y, fontsize=10.8)
    ax.set_title(panel, fontsize=17, fontweight='bold')
    ax.set_xlabel('Interaction coefficient')
    ax.grid(True, axis='x', alpha=0.20)
    ax.tick_params(axis='x', labelsize=11)

fig.suptitle('Contextual moderation of subnational social effects', fontsize=23, fontweight='bold', y=0.992)
legend_handles = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#333333', markeredgecolor='white', markersize=9, label='Global'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#cccccc', markeredgecolor='white', markersize=9, label='Region estimate'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='0.25', markeredgewidth=2, markersize=9, label='High-risk region (<10 countries)'),
    plt.Line2D([0], [0], color='#c9252d', marker='*', linestyle='None', markersize=10, label='FDR significant'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=4, frameon=False, fontsize=12, bbox_to_anchor=(0.5, 0.018))
fig.text(0.015, 0.006, 'Exact p and q values are reported in Tables. Global q-values are corrected across 4 interactions; region q-values across 28 region-interaction tests. High-risk regions are shown with hollow markers.', fontsize=11.2)
fig.tight_layout(rect=[0, 0.065, 1, 0.965])
fig.savefig(figures_dir / 'Figure_RQ4_1_contextual_moderation_global_region_coefficients.png', dpi=300)
plt.close(fig)

print('Integrated Figure RQ4 written.')
print(figures_dir / 'Figure_RQ4_1_contextual_moderation_global_region_coefficients.png')


## 6. Manifest

In [ ]:

manifest = {
    'input_file': str(data_path),
    'notebook': str(project / 'RQ4_contextual_moderation_main.ipynb'),
    'results_dir': str(results_dir),
    'process_dir': str(process_dir),
    'model_unit': 'GDL/subnational unit',
    'outcome': outcome,
    'fixed_effects': {'global_model': 'region fixed effects', 'region_models': 'none; models are subset by region'},
    'standard_errors': 'country-clustered SE; fallback noted if a model cannot compute clustered SE',
    'social_predictors': social_vars,
    'context_moderators': context_vars,
    'controls': controls,
    'interactions': interaction_defs,
    'sample': {'n_obs': int(len(sample)), 'n_countries': int(sample[country_col].nunique()), 'n_regions': int(sample[region_col].nunique())},
    'outputs': {
        'tables': sorted([p.name for p in tables_dir.glob('*.csv')]),
        'figures': sorted([p.name for p in figures_dir.glob('*.png')]),
        'process_files': sorted([p.name for p in process_dir.glob('*') if p.is_file()]),
    }
}
with open(process_dir / 'rq4_main_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))
